In [ ]:
# !pip install -U langchain-community
# !pip install -qU langchain-openai
# !pip install --upgrade langchain-community duckduckgo-search
# !pip install langgraph
# !pip install requests pubchempy rdkit-pypi
# !pip install ctx-python
# !pip install biopython
# !pip install langgraph-checkpoint-sqlite
# !pip install faiss-cpu
# !pip install langchain_openai python-dotenv
# !pip install "numpy<2"  # Use NumPy 1.x for compatibility with rdkit-pypi
# !pip install beautifulsoup4
# !pip install elsapy
# !pip install --upgrade numpy
# !pip install docling
# !pip install langchain-core==1.0.0
# !pip install --upgrade langchain
# !pip install sentence-transformers

# !pip install pdfplumber
# !pip install PyMuPDF
# !pip install --upgrade rdkit
# !pip install --upgrade --force-reinstall numpy spacy thinc flair
# !pip install docling docling-parse docling-core rapidocr-onnxruntime onnxruntime biopython requests pydantic bs4
# !pip install httpx[http2]
# !pip install pyab3p
# !pip install springernature-api-client
# !pip install lancedb

In [1]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr


load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)



/home/chen/GNPS_SIM/agent/db_form_agent_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# -------------------- stdlib --------------------
from pathlib import Path
import re, logging, warnings, sqlite3, os, time, random, pickle, requests
import json
import copy
from typing import List, TypedDict, Optional, Dict

# -------------------- langchain / langgraph --------------------
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.runnables import RunnableConfig

# -------------------- local tools --------------------
import sys
sys.path.append('../code')
from compound_tool import CompoundDatabaseTool
from searching_tool import FullTextJournalSearchTool
from fulltext_vector_tools import FullTextIndexTool, FullTextQueryTool


# -------------------- logging --------------------
logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(name)s | %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
log = logging.getLogger("batch_runner")

# -------------------- constants --------------------
MAX_ITERATIONS = 100

def _vprint(*args, **kwargs):
    if os.getenv("NODE_LOGS", "1") in ("1", "true", "TRUE", "yes", "YES"):
        print(*args, **kwargs)

# -------------------- prompts --------------------
PROMPTS_DIR = Path("/home/chen/GNPS_SIM/agent/db_form_agent_system/prompts")
LITERATURE_PROMPT_PATH = PROMPTS_DIR / "literature_agent.md"
CRITIQUE_PROMPT_PATH   = PROMPTS_DIR / "critique_agent.md"

def _load_prompt(path: Path, fallback: str) -> SystemMessage:
    try:
        return SystemMessage(content=path.read_text(encoding="utf-8"))
    except Exception:
        return SystemMessage(content=fallback.strip())

LITERATURE_AGENT_PROMPT = _load_prompt(
    LITERATURE_PROMPT_PATH,
    fallback="""
You are a meticulous research agent. Use tools when needed. Return JSON with a top-level "synthesis" object
including keys: claim (string), confidence_score (0..1), explanation (string), limitations (string), supporting_items (list).
""",
)

CRITIQUE_AGENT_PROMPT = _load_prompt(
    CRITIQUE_PROMPT_PATH,
    fallback="""
You are a critical evaluator. Read the researcher's output and decide if we should stop. Return JSON with keys:
decision ("approve" | "clarify" | "reject"), stop_reason (APPROVED | NO_NEW_EVIDENCE | STAGNANT_CLAIM | TOOL_UNAVAILABLE | FUTILITY_DETECTED | ""), 
supporting_sources (list), tool_name (string), agreement_score (0..1).
""",
)

# -------------------- model helpers --------------------
def _init_models():
    research_model = ChatOpenRouter(model_name="google/gemini-2.5-flash", temperature=0.0)
    evaluator_model = ChatOpenRouter(model_name="openai/gpt-4o-mini", temperature=0.0)
    return research_model, evaluator_model

# -------------------- tool factory --------------------
def _build_tools(results_base_dir: str, min_date: str = None, max_date: str = None) -> List:
    """
    Build tools for the agent.
    
    Args:
        results_base_dir: Base directory for search results
        min_date: Minimum publication date (YYYY, YYYY/MM, or YYYY/MM/DD format)
        max_date: Maximum publication date (YYYY, YYYY/MM, or YYYY/MM/DD format)
    
    Returns:
        List of tools including search, index, query, and compound database tools
    """
    ftjs = FullTextJournalSearchTool(
        results_base_dir=results_base_dir,
        include_pubmed=True,
        include_preprints=True,
        include_arxiv=True,
        include_medrxiv=True,
        include_eupmc=True,
        use_pubmed_free_fulltext_bias=True,
        max_parallel_downloads=6,
        min_date=min_date,  # Date range filter
        max_date=max_date,  # Date range filter
    )
    tool_list = [ftjs, FullTextIndexTool(results_base_dir=results_base_dir), FullTextQueryTool(results_base_dir=results_base_dir)]
    if CompoundDatabaseTool is not None:
        tool_list.insert(0, CompoundDatabaseTool())
    return tool_list

# -------------------- build agents --------------------
memory = MemorySaver()
research_model, evaluator_model = _init_models()
tools = _build_tools("/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results")

agent_executor = create_react_agent(
    research_model, tools, prompt=LITERATURE_AGENT_PROMPT, checkpointer=memory
)
critique_agent_executor = create_react_agent(
    evaluator_model, tools, prompt=CRITIQUE_AGENT_PROMPT, checkpointer=memory
)


# -------------------- build agents --------------------
memory = MemorySaver()
research_model, evaluator_model = _init_models()
tools = _build_tools("/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results")

agent_executor = create_react_agent(
    research_model, tools, prompt=LITERATURE_AGENT_PROMPT, checkpointer=memory
)
critique_agent_executor = create_react_agent(
    evaluator_model, tools, prompt=CRITIQUE_AGENT_PROMPT, checkpointer=memory
)

# -------------------- state definitions --------------------
class StateHistory(TypedDict):
    claim: str
    confidence_score: float
    supporting_sources: List[str]
    tool_name: str | None
    stop_reason: str | None

class AgentState(TypedDict):
    messages: List[HumanMessage | AIMessage]
    original_query: str
    search_session_dir: str
    stop_history: List[StateHistory]
    consecutive_no_new_evidence: int
    consecutive_tool_unavailable: int
    last_researcher_json: str | None
    last_critique_json: str | None
    claim_history: List[str]
    confidence_history: List[float]

# -------------------- helpers --------------------
def _last_researcher_json(messages):
    if not messages:
        return ""
    for msg in reversed(messages):
        content = getattr(msg, "content", "") or ""
        if '"synthesis"' in content:
            return content
    return ""

def determine_final_claim(messages):
    blob = _last_researcher_json(messages)
    if not blob:
        return "Unclear"
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        claim_raw = syn.get("claim")
        if isinstance(claim_raw, str) and claim_raw.strip():
            return claim_raw.strip()
    except Exception:
        pass
    m = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
    if m:
        candidate = m.group(1).strip()
        if candidate:
            return candidate
    return "Unclear"

def _safe_float(x):
    try:
        if isinstance(x, (int, float)):
            return float(x)
        if isinstance(x, str):
            return float(x.replace("–", "-").strip())
    except Exception:
        return None

def determine_confidence_score(messages):
    blob = _last_researcher_json(messages)
    if not blob:
        return 0.0
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        score = syn.get("confidence_score")
        return _safe_float(score) or 0.0
    except Exception:
        pass
    m = re.search(r'"confidence_score"\s*:\s*([0-9.]+)', blob)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            pass
    return 0.0

def determine_explanation(messages):
    blob = _last_researcher_json(messages)
    if not blob:
        return ""
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        explanation = syn.get("explanation")
        if isinstance(explanation, str) and explanation.strip():
            return explanation.strip()
    except Exception:
        pass
    m = re.search(r'"explanation"\s*:\s*"([^"]+)"', blob)
    if m:
        return m.group(1).strip()
    return ""

# -------------------- nodes --------------------
def researcher_node(state: AgentState, config: RunnableConfig):
    _vprint("-----Researcher Node Invoked-----")

    crit_json = state.get("last_critique_json")
    prev_research_json = state.get("last_researcher_json") or _last_researcher_json(state.get("messages", []))

    if crit_json and prev_research_json:
        next_query_val = None
        follow_up_str = ""
        try:
            crit_obj = json.loads(crit_json) if isinstance(crit_json, str) else (crit_json or {})
            critique_section = crit_obj.get("critique", {})
            if not isinstance(critique_section, dict):
                critique_section = {}
            decision = critique_section.get("decision", "")
            if isinstance(decision, str) and decision.lower() == "clarify":
                next_query_val = critique_section.get("next_query")
            if next_query_val:
                if isinstance(next_query_val, list):
                    query_str = "\n".join(str(item) for item in next_query_val if item)
                elif isinstance(next_query_val, str):
                    query_str = next_query_val.strip()
                else:
                    query_str = ""
                if query_str:
                    follow_up_str = f"\nReviewer follow-up request:\n{query_str}\n\n"
        except Exception as e:
            _vprint(f"[WARN] Error parsing critique JSON: {e}")

        merged = (
            "Previous researcher output:\n"
            f"{prev_research_json}\n\n"
            "Reviewer critique JSON:\n"
            f"{crit_json}\n\n"
            + follow_up_str
        )
        input_messages = [HumanMessage(content=merged)]
    else:
        input_messages = state["messages"]

    session_dir = state.get("search_session_dir")
    if not session_dir:
        raise ValueError("search_session_dir not found in state.")

    tool_config = config.copy()
    tool_config["configurable"] = config.get("configurable", {}).copy()
    tool_config["configurable"]["search_session_dir"] = session_dir

    # (tags/metadata optional)
    tags = list(tool_config.get("tags") or [])
    tags.append(f"SSD::{session_dir}")
    tags.append(f"search_session_dir={session_dir}")
    tool_config["tags"] = tags

    metadata = dict(tool_config.get("metadata") or {})
    metadata["search_session_dir"] = session_dir
    tool_config["metadata"] = metadata

    tool_config.pop("recursion_limit", None)

    # ---- Seed ContextVar for all tools (authoritative path propagation)
    try:
        FullTextJournalSearchTool.set_session_dir(session_dir)
        FullTextIndexTool.set_session_dir(session_dir)
        FullTextQueryTool.set_session_dir(session_dir)
    except Exception:
        pass

    result = agent_executor.invoke({"messages": input_messages}, config=tool_config)

    ai_content = ""
    if 'messages' in result and isinstance(result['messages'], list) and result['messages']:
        ai_content = result['messages'][-1].content
    else:
        _vprint("[ERROR] Agent executor did not return expected messages list.")
        ai_content = (
            '{"synthesis": {"claim": "Unclear", "explanation": "Agent execution failed internally. '
            'No message was returned.", "confidence_score": 0.0, "limitations": "Agent internal error."}, '
            '"traceability": {}, "next_actions": {}}'
        )

    new_ai_message = AIMessage(content=ai_content)
    all_messages = state["messages"] + [new_ai_message]

    return {
        "messages": all_messages,
        "last_researcher_json": _last_researcher_json(all_messages),
        "last_critique_json": None,
    }

def critique_node(state: AgentState, config: RunnableConfig):
    _vprint("-----Critique Node Invoked-----")

    last_researcher_json = _last_researcher_json(state.get("messages", [])) or ""
    if not last_researcher_json:
        last_researcher_json = (state["messages"][-1].content if state.get("messages") else "").strip()

    critique_input = f"Researcher output JSON:\n{last_researcher_json}"

    session_dir = state.get("search_session_dir")
    if not session_dir:
        raise ValueError("search_session_dir not found in state.")

    tool_config = config.copy()
    tool_config["configurable"] = config.get("configurable", {}).copy()
    tool_config["configurable"]["search_session_dir"] = session_dir

    tags = list(tool_config.get("tags") or [])
    tags.append(f"SSD::{session_dir}")
    tags.append(f"search_session_dir={session_dir}")
    tool_config["tags"] = tags

    metadata = dict(tool_config.get("metadata") or {})
    metadata["search_session_dir"] = session_dir
    tool_config["metadata"] = metadata

    tool_config.pop("recursion_limit", None)

    # Seed ContextVar for tools here as well
    try:
        FullTextJournalSearchTool.set_session_dir(session_dir)
        FullTextIndexTool.set_session_dir(session_dir)
        FullTextQueryTool.set_session_dir(session_dir)
    except Exception:
        pass

    critique_result = critique_agent_executor.invoke({"messages": [HumanMessage(content=critique_input)]}, config=tool_config)

    critique_content = critique_result['messages'][-1].content
    raw = (critique_content or "").strip()

    parsed: dict | None = None
    try:
        candidate = json.loads(raw)
        if isinstance(candidate, dict):
            parsed = candidate
    except Exception:
        pass
    if parsed is None:
        match = re.search(r"\{[\s\S]*?\}", raw)
        if match:
            try:
                candidate = json.loads(match.group(0))
                if isinstance(candidate, dict):
                    parsed = candidate
            except Exception:
                parsed = None

    stop_reason = None
    supporting_sources: List[str] = []
    tool_name: str | None = None

    if parsed:
        candidate_reason = parsed.get("stop_reason")
        if isinstance(candidate_reason, str) and candidate_reason.strip():
            stop_reason = candidate_reason.strip().upper()
        sources = parsed.get("supporting_sources")
        if isinstance(sources, list):
            supporting_sources = [str(item) for item in sources]
        candidate_tool = parsed.get("tool_name")
        if isinstance(candidate_tool, str) and candidate_tool.strip():
            tool_name = candidate_tool.strip()

    def _extract_claim_and_confidence(messages):
        blob = _last_researcher_json(messages)
        claim = "Unclear"
        confidence = 0.0
        if not blob:
            return claim, confidence
        try:
            obj = json.loads(blob)
            synthesis = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
            claim = (synthesis.get("claim") or "").strip() if isinstance(synthesis, dict) else ""
            candidate_conf = _safe_float(synthesis.get("confidence_score")) if isinstance(synthesis, dict) else None
            if candidate_conf is not None:
                confidence = max(0.0, min(1.0, candidate_conf))
        except Exception:
            pass
        if not claim:
            match_claim = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
            if match_claim:
                claim = match_claim.group(1).strip()
        if confidence == 0.0:
            match_conf = re.search(r'"confidence_score"\s*:\s*("?)([0-9]*\.?[0-9]+)\1', blob)
            if match_conf:
                try:
                    cand = float(match_conf.group(2))
                    confidence = max(0.0, min(1.0, cand))
                except Exception:
                    pass
        return claim or "Unclear", confidence

    claim, confidence = _extract_claim_and_confidence(state.get("messages", []))

    history = copy.deepcopy(state.get("stop_history", []))
    history.append({
        "claim": claim,
        "confidence_score": confidence,
        "supporting_sources": supporting_sources,
        "tool_name": tool_name,
        "stop_reason": stop_reason,
    })

    prev_no_new = state.get("consecutive_no_new_evidence", 0)
    prev_tool_unavailable = state.get("consecutive_tool_unavailable", 0)
    no_new = prev_no_new

    claim_hist = list(state.get("claim_history", []))
    conf_hist = list(state.get("confidence_history", []))
    claim_hist.append(claim if isinstance(claim, str) else "Unclear")
    conf_hist.append(float(confidence) if confidence is not None else 0.0)

    STABLE_THRESHOLD = 3
    CONFIDENCE_EPSILON = 0.001
    if len(claim_hist) >= STABLE_THRESHOLD and len(conf_hist) >= STABLE_THRESHOLD:
        last_n_claims = claim_hist[-STABLE_THRESHOLD:]
        last_n_confs = conf_hist[-STABLE_THRESHOLD:]
        normalized_claims = [c.strip().lower() if isinstance(c, str) else "" for c in last_n_claims]
        all_same_claim = len(set(normalized_claims)) == 1
        conf_range = max(last_n_confs) - min(last_n_confs)
        all_same_conf = conf_range <= CONFIDENCE_EPSILON
        if all_same_claim and all_same_conf and not stop_reason:
            stop_reason = "CONSECUTIVE_SAME_RESULT"
            _vprint(f"[critique_node] Detected {STABLE_THRESHOLD} consecutive identical results: {last_n_claims[-1]} @ {last_n_confs[-1]:.3f}")
            if history:
                history[-1]["stop_reason"] = stop_reason

    _vprint(f"[critique_node] stop_reason={stop_reason}")

    patched_content = critique_content
    if parsed is not None:
        patched = dict(parsed)
        if stop_reason:
            patched["stop_reason"] = stop_reason
        patched_content = json.dumps(patched, indent=2)

    return {
        "messages": state["messages"] + [AIMessage(content=patched_content)],
        "stop_history": history,
        "consecutive_no_new_evidence": no_new,
        "consecutive_tool_unavailable": prev_tool_unavailable,
        "last_researcher_json": last_researcher_json,
        "last_critique_json": patched_content,
        "claim_history": claim_hist,
        "confidence_history": conf_hist,
    }

def should_continue(state: AgentState):
    last_message_content = state["messages"][-1].content if state.get("messages") else ""
    raw = (last_message_content or "").strip()
    _vprint(f"[should_continue] raw critique output: ...")

    STOP_REASONS = {"APPROVED","NO_NEW_EVIDENCE","TOOL_UNAVAILABLE","CONSECUTIVE_SAME_RESULT"}

    def _normalized_reason(value) -> str | None:
        if isinstance(value, str) and value.strip():
            return value.strip().upper()
        return None

    history = state.get("stop_history") or []
    if history:
        latest_reason = _normalized_reason(history[-1].get("stop_reason"))
        if latest_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (history stop_reason={latest_reason})-----")
            return END

    def _should_stop_from_json(obj: dict) -> bool:
        stop_reason = _normalized_reason(obj.get("stop_reason"))
        if stop_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (stop_reason={stop_reason})-----")
            return True
        decision = obj.get("decision")
        if isinstance(decision, str) and decision.lower() == "approve":
            _vprint("-----Agent Decided to Stop (legacy decision=approve)-----")
            return True
        try:
            score = float(obj.get("agreement_score", 0.0))
            if score > 0.75:
                _vprint(f"-----Agent Decided to Stop (legacy agreement_score={score:.2f} > 0.75)-----")
                return True
        except Exception:
            pass
        return False

    try:
        critique_json = json.loads(raw)
        if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
            return END
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*?\}", raw)
    if match:
        try:
            critique_json = json.loads(match.group(0))
            if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
                _vprint("-----Agent Decided to Stop (extracted JSON)-----")
                return END
        except Exception:
            pass

    low = raw.lower()
    if any(token in low for token in ["approve", "approved"]):
        _vprint("-----Agent Decided to Stop (text match)-----")
        return END

    return "researcher"

# -------------------- workflow --------------------
workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("critique", critique_node)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "critique")
workflow.add_conditional_edges("critique", should_continue)

app = workflow.compile(checkpointer=memory)



/tmp/ipykernel_371822/3020205449.py:92: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(
/tmp/ipykernel_371822/3020205449.py:95: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  critique_agent_executor = create_react_agent(


In [3]:


# def main():
#     _vprint("\n🧪 Analytical Chemistry Agent Ready!")
#     while True:
#         try:
#             query = input("\n📝 Enter your question (or 'quit' to exit): ")
#         except (EOFError, KeyboardInterrupt):
#             _vprint("\nExiting.")
#             break

#         _vprint(f"\n>>> User input query: {query}\n", flush=True)
#         if query.lower().strip() == 'quit':
#             break

#         # per-run session dir for all tools to reuse
#         query_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
#         query_results_dir = f"/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_{query_timestamp}"
#         os.environ["SEARCH_SESSION_DIR"] = query_results_dir

#         # optional tracing/project metadata
#         os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
#         os.environ["LANGCHAIN_PROJECT"] = f"DB Form Agent Run - {query_timestamp}"
#         _vprint(f"📁 All results will be saved to: searching_results/run_{query_timestamp}\n")

#         init_message = HumanMessage(content=query)
#         init_state = {
#             'messages': [init_message],
#             'original_query': query,
#             'stop_history': [],
#             'consecutive_no_new_evidence': 0,
#             'consecutive_tool_unavailable': 0,
#             'claim_history': [],
#             'confidence_history': [],
#         }

#         thread_id = f"qa_session_{int(time.time())}"
#         last_event_state = None

#         try:
#             for event in app.stream(
#                 init_state,
#                 config={'configurable': {'thread_id': thread_id}, 'recursion_limit': MAX_ITERATIONS}
#             ):
#                 for v in event.values():
#                     last_event_state = v
#                     try:
#                         content = v['messages'][-1].content
#                         try:
#                             obj = json.loads(content)
#                             _vprint(json.dumps(obj, indent=2))
#                         except json.JSONDecodeError:
#                             _vprint(content)
#                     except Exception:
#                         _vprint(str(v))
#         except GraphRecursionError as ge:
#             _vprint(f"[GraphRecursionError] {ge}")
#         except KeyboardInterrupt:
#             _vprint("\nInterrupted by user.")
#             break
#         except Exception as e:
#             _vprint(f"[ERROR] Agent execution failed: {e}")

#         claim = 'Unclear'
#         score = 0.0
#         explanation = ""
#         critique_decision = None
#         critique_json = {}

#         if last_event_state and last_event_state.get('messages'):
#             claim = determine_final_claim(last_event_state['messages'])
#             score = determine_confidence_score(last_event_state['messages']) or 0.0
#             explanation = determine_explanation(last_event_state['messages'])

#         display_label = claim.strip() if isinstance(claim, str) and claim.strip() else "Unclear"

#         final_stop_reason = None
#         if last_event_state and last_event_state.get('stop_history'):
#             final_stop_reason = (last_event_state['stop_history'][-1] or {}).get('stop_reason')

#         _vprint(f"\n=== FINAL DECISION: {display_label} (confidence {score:.2f}) ===")
#         if final_stop_reason:
#             _vprint(f"    stop reason: {final_stop_reason}")
#         if explanation:
#             _vprint(f"    explanation: {explanation}")

#         if last_event_state and last_event_state.get('messages'):
#             critique_msg = last_event_state['messages'][-1]
#             critique_raw = getattr(critique_msg, 'content', '') or ''
#             try:
#                 critique_json = json.loads(critique_raw)
#             except Exception:
#                 match_json = re.search(r"\{[\s\S]*?\}", critique_raw)
#                 if match_json:
#                     try:
#                         critique_json = json.loads(match_json.group(0))
#                     except Exception:
#                         critique_json = {}
#             if isinstance(critique_json, dict):
#                 decision_val = critique_json.get('decision')
#                 if isinstance(decision_val, str):
#                     critique_decision = decision_val.strip().lower()
#         if critique_decision:
#             _vprint(f"    critique decision: {critique_decision}")
#         _vprint(f"    researcher claim: {claim}")


# if __name__ == '__main__':
#     main()

In [3]:


# -------------------- main: parallel batch --------------------
def main():
    _vprint("\n🧪 Analytical Chemistry Agent Ready! (Parallel Batch Mode)")

    # import pandas as pd

    # df = pd.read_csv("/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/quantified_metabolites.csv")

    # # get 100 rows with 80% endogenous
    # df_endogenous = df[df['Endogenous']=='Yes'].sample(n=27, random_state=42)
    # df_non_endogenous = df[df['Endogenous']=='No'].sample(n=3, random_state=42)
    # df_sampled = pd.concat([df_endogenous, df_non_endogenous]).reset_index(drop=True)

    # input_command = df_sampled["Name"].to_list()

    # queries_to_run = []
    # for item in input_command:
    #     queries_to_run += [f'is the compound {item} endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"']

    queries_to_run = [
        # 'is the compound DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0) endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"',
        # '''is S-4'-Hydroxywarfarin endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"''',
        # 'is the compound biotin endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"'
        'is compound biotin toxic to humans? Answer: "Toxic" or "Non-Toxic"',
        'is compound ricin toxic to humans? Answer: "Toxic" or "Non-Toxic"',
        # 'is the compound biotin endogenous metabolite generated by human body? Answer: "Endogenous" or "Exogenous"'

    ]
    _vprint(f"Found {len(queries_to_run)} queries to run in parallel.")

    # optional tracing/project metadata
    time_run = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
    os.environ["LANGCHAIN_PROJECT"] = f"DB Form Agent Run - {time_run}"

    all_init_states: List[AgentState] = []
    all_configs: List[RunnableConfig] = []

    base_results_dir = "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results"

    for query in queries_to_run:
        query_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
        micros = f"{int(time.time_ns() % 1_000_000):06d}"
        nonce = f"{random.randint(0, 9999):04d}"

        query_results_dir_rel = f"run_{query_timestamp}_{micros}_{nonce}"

        query_results_dir_abs = f"{base_results_dir}/{query_results_dir_rel}"

        _vprint(query_results_dir_abs)
        os.makedirs(query_results_dir_abs, exist_ok=True)

        init_state: AgentState = {
            'messages': [HumanMessage(content=query)],
            'original_query': query,
            'search_session_dir': query_results_dir_abs,
            'stop_history': [],
            'consecutive_no_new_evidence': 0,
            'consecutive_tool_unavailable': 0,
            'last_researcher_json': None,
            'last_critique_json': None,
            'claim_history': [],
            'confidence_history': [],
        }
        all_init_states.append(init_state)

        thread_id = f"qa_session_{query_timestamp}_{random.randint(1000, 9999)}"
        cfg: RunnableConfig = {
            'configurable': {
                'thread_id': thread_id,
                'search_session_dir': query_results_dir_abs,  # still provided (not relied upon)
            },
            'recursion_limit': MAX_ITERATIONS,
            "max_concurrency": 2,
        }
        all_configs.append(cfg)

        # (Optional) Also seed ContextVar here (belt-and-suspenders)
        try:
            FullTextJournalSearchTool.set_session_dir(query_results_dir_abs)
            FullTextIndexTool.set_session_dir(query_results_dir_abs)
            FullTextQueryTool.set_session_dir(query_results_dir_abs)
        except Exception:
            pass

    all_results_data = []

    try:
        start_time = time.time()
        _vprint(f"Running {len(all_init_states)} queries in parallel...")
        final_states = app.batch(
            all_init_states,
            all_configs,
        )
        end_time = time.time()

        _vprint(f"\n{'='*20}\n✅ BATCH COMPLETE\n{'='*20}")
        _vprint(f"Total time: {end_time - start_time:.2f} seconds")
        _vprint(f"Total queries processed: {len(final_states)}")

        for final_state in final_states:
            if not final_state:
                all_results_data.append({
                    "query": "UNKNOWN (State was None)",
                    "claim": "Error",
                    "explanation": "Batch run returned a None state.",
                    "stop_reason": "BATCH_ERROR",
                    "results_dir": None
                })
                continue

            query = final_state.get('original_query', 'Unknown Query')
            abs_path = final_state.get('search_session_dir', '')
            rel_path = abs_path.replace(base_results_dir + '/', '') if abs_path else ''

            claim = "Error"
            score = 0.0
            explanation = "Batch run failed to produce messages."
            stop_reason = "BATCH_ERROR"

            if final_state.get('messages'):
                claim = determine_final_claim(final_state['messages'])
                score = determine_confidence_score(final_state['messages']) or 0.0
                explanation = determine_explanation(final_state['messages'])

            if final_state.get('stop_history'):
                stop_reason = (final_state['stop_history'][-1] or {}).get('stop_reason', 'COMPLETED')

            all_results_data.append({
                "query": query,
                "claim": claim,
                "confidence_score": score,
                "explanation": explanation,
                "stop_reason": stop_reason,
                "results_dir": f"searching_results/{rel_path}" if rel_path else None
            })

        print("\n--- Batch Run Summary ---")
        print(json.dumps(all_results_data, indent=2))

        summary_path = f"{base_results_dir}/batch_summary.json"
        try:
            with open(summary_path, "w") as f:
                json.dump(all_results_data, f, indent=2)
            _vprint(f"\nBatch summary saved to: {summary_path}")
        except Exception as e:
            _vprint(f"\n[ERROR] Could not save batch summary to {summary_path}: {e}")

    except KeyboardInterrupt:
        _vprint("\nBatch run interrupted by user.")
    except GraphRecursionError as ge:
        _vprint(f"[GraphRecursionError] {ge}")
    except Exception as e:
        _vprint(f"[ERROR] Agent execution failed: {e}")

    _vprint("\nBatch processing finished.")

if __name__ == '__main__':
    main()




🧪 Analytical Chemistry Agent Ready! (Parallel Batch Mode)
Found 2 queries to run in parallel.
/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251105_212611_200492_3695
/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251105_212611_825950_4183
Running 2 queries in parallel...
-----Researcher Node Invoked-----
-----Researcher Node Invoked-----


2025-11-05 13:26:13,025 | INFO | PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2025-11-05 13:26:14,088 | INFO | Using session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251105_212611_825950_4183
2025-11-05 13:26:15,277 | INFO | Using session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/run_20251105_212611_200492_3695
2025-11-05 13:26:28,250 | INFO | Found 33 results total, 19 unique. Enriching...
2025-11-05 13:26:28,251 | INFO | Raw source counts before deduplication: {'EuropePMC:MED': 15, 'ChemRxiv': 3, 'PubMed': 15}
2025-11-05 13:26:29,140 | INFO | Docling URL conversions: 8 candidates (max_workers=6)
2025-11-05 13:26:31,129 | INFO | detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-05 13:26:31,149 | INFO | Going to convert document batch...
2025-11-05 13:26:31,150 | INFO | Initializing pipeline for StandardPdfPipeline with options hash 4f2edc0f7d9bb60b38ebfecf9

-----Critique Node Invoked-----
[critique_node] stop_reason=None
[should_continue] raw critique output: ...
-----Agent Decided to Stop (text match)-----


2025-11-05 13:31:03,636 | INFO | Finished converting document ja5c07109.pdf in 32.97 sec.
2025-11-05 13:31:03,680 | INFO | [Docling OK] EuropePMC:MED: Potent Inducers of Paraptosis through Electronic Tuning of H
2025-11-05 13:31:06,388 | INFO | detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-05 13:31:06,406 | INFO | Going to convert document batch...
2025-11-05 13:31:06,417 | INFO | Processing document thnov15p9742.pdf
2025-11-05 13:31:11,918 | INFO | Finished converting document TID-23-132.pdf in 29.19 sec.
2025-11-05 13:31:11,996 | INFO | [Docling OK] EuropePMC:MED: Mouse model validity for studying the impact of tobacco smok
2025-11-05 13:31:12,290 | WARNING | [Docling EMPTY] EuropePMC:MED: Phthalocyanines Conjugated with Small Biologically Active Co
2025-11-05 13:31:15,208 | INFO | detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-05 13:31:15,316 | INFO | Going to convert document batch...
2025-11-05 13:31:15,317 | INFO | Processing document biomedicines-13-02545.pdf
2025-1

-----Critique Node Invoked-----
[critique_node] stop_reason=None
[should_continue] raw critique output: ...
-----Agent Decided to Stop (text match)-----

✅ BATCH COMPLETE
Total time: 521.69 seconds
Total queries processed: 2

--- Batch Run Summary ---
[
  {
    "query": "is compound biotin toxic to humans? Answer: \"Toxic\" or \"Non-Toxic\"",
    "claim": "Non-Toxic",
    "confidence_score": 0.0,
    "explanation": "Biotin appears to be relatively non-toxic to humans, even at high doses (e.g., 60 mg/day for several months). The FDA has issued warnings regarding biotin's interference with laboratory tests, leading to inaccurate measurements and potential clinical misinterpretations, but not due to direct toxicity to the human body.",
    "stop_reason": null,
    "results_dir": "searching_results/run_20251105_212611_200492_3695"
  },
  {
    "query": "is compound ricin toxic to humans? Answer: \"Toxic\" or \"Non-Toxic\"",
    "claim": "Toxic",
    "confidence_score": 0.9,
    "explanatio

In [6]:
# from the batch, extract the claim and compound name from /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/batch_summary.json

loaded_summary_path = "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/batch_summary.json"
with open(loaded_summary_path, "r") as f:
    batch_data = json.load(f)
compound_claims = []
for entry in batch_data:
    query = entry.get("query", "")
    claim = entry.get("claim", "")
    m = re.search(r'is the compound (.+?) endogenous metabolite from human\?', query)
    compound_name = m.group(1).strip() if m else "Unknown Compound"
    compound_claims.append({
        "compound": compound_name,
        "claim": claim
    })

In [8]:
import pandas as pd

df = pd.read_csv("/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/quantified_metabolites.csv")

# get 100 rows with 80% endogenous
df_endogenous = df[df['Endogenous']=='Yes'].sample(n=27, random_state=42)
df_non_endogenous = df[df['Endogenous']=='No'].sample(n=3, random_state=42)
df_sampled = pd.concat([df_endogenous, df_non_endogenous]).reset_index(drop=True)

In [9]:
# Create a DataFrame from the extracted compound claims
df_compound_claims = pd.DataFrame(compound_claims)

# Merge the original DataFrame with the compound claims DataFrame
df_merged = df_sampled.merge(df_compound_claims, left_on="Name", right_on="compound", how="left")


In [13]:
df_merged[['Name', 'Endogenous', 'claim']]

,Name,Endogenous,claim
0,"DG(20:3(5Z,8Z,11Z)/20:3(5Z,8Z,11Z)/0:0)",Yes,Unclear
1,"TG(16:0/16:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))",Yes,Plausible Endogenous
2,"PC(18:2(9Z,12Z)/22:2(13Z,16Z))",Yes,Unclear
3,Dihydrofolic acid,Yes,Endogenous
4,"DG(22:2(13Z,16Z)/16:0/0:0)",Yes,Unclear
5,"8,9-DiHETrE",Yes,Plausible Endogenous
6,L-Kynurenine,Yes,Endogenous
7,"PC(18:3(9Z,12Z,15Z)/P-18:1(11Z))",Yes,Unclear
8,3-Hydroxy-9Z-octadecenoylcarnitine,Yes,Unclear
9,Pseudouridine,Yes,Endogenous


In [12]:
df_merged[['Name', 'Endogenous', 'claim']].to_csv("/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/agent_compound_claims_results.csv", index=False)

In [5]:
# Direct tool test
def direct_tool_test():
    _vprint("\\n🔬 Directly testing FullTextJournalSearchTool...")
    
    queries_to_test = {
        "medrxiv": "COVID-19",
        "chemrxiv": "catalysis"
    }
    
    for source, test_query in queries_to_test.items():
        max_results = 5
        
        test_run_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
        test_results_dir = f"/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_{source}_{test_run_timestamp}"
        os.makedirs(test_results_dir, exist_ok=True)
        
        _vprint(f"\\n--- Testing {source.upper()} with query: '{test_query}' ---")
        _vprint(f"Results will be saved to: {test_results_dir}")

        tool = FullTextJournalSearchTool(results_base_dir=test_results_dir)
        FullTextJournalSearchTool.set_session_dir(test_results_dir)

        try:
            # Enable only the source being tested
            run_kwargs = {
                "query": test_query,
                "max_results": max_results,
                "include_pubmed": False,
                "include_arxiv": False,
                "include_medrxiv": source == "medrxiv",
                "include_chemrxiv": source == "chemrxiv",
            }
            
            result_summary = tool._run(**run_kwargs)
            
            _vprint("\\n--- Test Summary ---")
            _vprint(result_summary)
            
            saved_files = list(Path(test_results_dir).glob('*.json'))
            if saved_files:
                _vprint(f"✅ {source.upper()} Success! Found {len(saved_files)} result files.")
            else:
                _vprint(f"❌ {source.upper()} Failure! No result files (.json) were saved.")

        except Exception as e:
            _vprint(f"\\n[ERROR] {source.upper()} test failed: {e}")

# Run the direct test
direct_tool_test()


2025-10-30 13:53:59,273 | INFO | Using session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | Starting search #1/1 for query: 'COVID-19'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | [1/1] Searching medRxiv…
2025-10-30 13:53:59,275 | INFO | Starting search #1/1 for query: 'COVID-19'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | [1/1] Searching medRxiv…


2025-10-30 13:53:59,273 | INFO | Using session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | Starting search #1/1 for query: 'COVID-19'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | [1/1] Searching medRxiv…
2025-10-30 13:53:59,275 | INFO | Starting search #1/1 for query: 'COVID-19'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | [1/1] Searching medRxiv…


\n🔬 Directly testing FullTextJournalSearchTool...
\n--- Testing MEDRXIV with query: 'COVID-19' ---
Results will be saved to: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359


2025-10-30 13:53:59,273 | INFO | Using session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | Starting search #1/1 for query: 'COVID-19'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | [1/1] Searching medRxiv…
2025-10-30 13:53:59,275 | INFO | Starting search #1/1 for query: 'COVID-19'; saving to /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
2025-10-30 13:53:59,275 | INFO | [1/1] Searching medRxiv…


\n🔬 Directly testing FullTextJournalSearchTool...
\n--- Testing MEDRXIV with query: 'COVID-19' ---
Results will be saved to: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359


2025-10-30 13:54:00,033 | INFO | Found 5 results total, 5 unique. Fetching full text...
2025-10-30 13:54:00,885 | INFO | [JATS OK] medRxiv: Eleven key measures for monitoring general practice clinical
2025-10-30 13:54:00,885 | INFO | [JATS OK] medRxiv: Eleven key measures for monitoring general practice clinical
2025-10-30 13:54:01,354 | INFO | [JATS OK] medRxiv: A novel hospital-at-home model for patients with COVID-19 bu
2025-10-30 13:54:01,354 | INFO | [JATS OK] medRxiv: A novel hospital-at-home model for patients with COVID-19 bu
2025-10-30 13:54:01,852 | INFO | [JATS OK] medRxiv: The impact of the COVID-19 pandemic on the reporting of viol
2025-10-30 13:54:01,852 | INFO | [JATS OK] medRxiv: The impact of the COVID-19 pandemic on the reporting of viol
2025-10-30 13:54:02,389 | INFO | [JATS OK] medRxiv: The Impact of COVID-19 on Life Expectancy among Asian Americ
2025-10-30 13:54:02,389 | INFO | [JATS OK] medRxiv: The Impact of COVID-19 on Life Expectancy among Asian Americ
2025-10-

... (11 outputs left out)

2025-10-30 13:54:00,033 | INFO | Found 5 results total, 5 unique. Fetching full text...
2025-10-30 13:54:00,885 | INFO | [JATS OK] medRxiv: Eleven key measures for monitoring general practice clinical
2025-10-30 13:54:00,885 | INFO | [JATS OK] medRxiv: Eleven key measures for monitoring general practice clinical
2025-10-30 13:54:01,354 | INFO | [JATS OK] medRxiv: A novel hospital-at-home model for patients with COVID-19 bu
2025-10-30 13:54:01,354 | INFO | [JATS OK] medRxiv: A novel hospital-at-home model for patients with COVID-19 bu
2025-10-30 13:54:01,852 | INFO | [JATS OK] medRxiv: The impact of the COVID-19 pandemic on the reporting of viol
2025-10-30 13:54:01,852 | INFO | [JATS OK] medRxiv: The impact of the COVID-19 pandemic on the reporting of viol
2025-10-30 13:54:02,389 | INFO | [JATS OK] medRxiv: The Impact of COVID-19 on Life Expectancy among Asian Americ
2025-10-30 13:54:02,389 | INFO | [JATS OK] medRxiv: The Impact of COVID-19 on Life Expectancy among Asian Americ
2025-10-

\n--- Test Summary ---
Total Papers Found: 5. Run_dir: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_medrxiv_20251030_205359
✅ MEDRXIV Success! Found 5 result files.
\n--- Testing CHEMRXIV with query: 'catalysis' ---
Results will be saved to: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_chemrxiv_20251030_205402


2025-10-30 13:54:03,666 | INFO | Found 0 results total, 0 unique. Fetching full text...
2025-10-30 13:54:03,667 | INFO | Filtered to 0 papers with full text (skipped 0 without)
2025-10-30 13:54:03,668 | INFO | No papers with full text found to save.
2025-10-30 13:54:03,669 | INFO | Completed 1 search run(s). Total saved files: 0
2025-10-30 13:54:03,667 | INFO | Filtered to 0 papers with full text (skipped 0 without)
2025-10-30 13:54:03,668 | INFO | No papers with full text found to save.
2025-10-30 13:54:03,669 | INFO | Completed 1 search run(s). Total saved files: 0


\n--- Test Summary ---
Total Papers Found: 0. Run_dir: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/direct_test_chemrxiv_20251030_205402
❌ CHEMRXIV Failure! No result files (.json) were saved.
